# Eddy Current Engine — Parameter Explorer

Edit only the parameter cell below to explore the three-disk geometry, phase drive, torque cycle, and reduced rotor transient. The stored torque coefficients correspond to the reference geometry; recompute them with `Coefficient_Integration.ipynb` after changing disk geometry if quantitative torque values are required.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np

from eddy_current_engine import (
    Disk, ElectromagneticParameters, GeometryParameters, analyze_geometry,
    cycle_average_torque, drive_geometry_factor, instantaneous_torque,
    normalized_rotor, phase_drive, reduced_terminal_speed,
)
from eddy_current_engine.visualization import (
    plot_geometry, plot_phase_response, plot_torque_cycle, plot_transient,
)

## Parameters

Lengths are in metres, magnetic fields in tesla, conductivity in S/m, phase in radians, and frequencies in hertz.

In [ ]:
# --- Geometry ---
DISK_1_CENTER = (0.000, 0.000)
DISK_1_RADIUS = 0.040
DISK_2_CENTER = (0.000, -0.009)
DISK_2_RADIUS = (0.02807 + 0.0398) / 2
DISK_3_CENTER = (0.021, -0.009)
DISK_3_RADIUS = 0.040
GEOMETRY_RESOLUTION = 256

# --- Electromagnetic drive ---
PHASE = 0.07
FREQUENCY_HZ = 60.0
CONDUCTIVITY = 31.95973e6
FIELD_1 = 0.006
FIELD_2 = 0.006
ROTOR_SPEED_FOR_TORQUE = 1.0

# --- Reduced rotor transient ---
TIME_CONSTANT = 0.20
FINAL_TIME = 1.0
N_POINTS = 600

In [ ]:
geometry_parameters = GeometryParameters(
    disk_1=Disk(DISK_1_CENTER, DISK_1_RADIUS),
    disk_2=Disk(DISK_2_CENTER, DISK_2_RADIUS),
    disk_3=Disk(DISK_3_CENTER, DISK_3_RADIUS),
)
analysis = analyze_geometry(geometry_parameters, resolution=GEOMETRY_RESOLUTION)

for name, measurement in analysis.measurements.items():
    print(f'{name:24s} area={measurement.area:.10g}  centroid={measurement.centroid}')

plot_geometry(analysis);

In [ ]:
em = ElectromagneticParameters(
    magnetic_field_1=FIELD_1, magnetic_field_2=FIELD_2,
    conductivity=CONDUCTIVITY, drive_frequency_hz=FREQUENCY_HZ,
    rotor_speed=ROTOR_SPEED_FOR_TORQUE, disk_radius=DISK_1_RADIUS,
)
m = analysis.measurements
factor = drive_geometry_factor(
    m['disk_1_disk_2_only'].centroid, m['triple'].centroid, DISK_1_RADIUS
)
phases = np.linspace(-np.pi, np.pi, N_POINTS)
plot_phase_response(phases, phase_drive(factor, phases), PHASE);
print('Complex geometry factor:', factor)

In [ ]:
period = 1.0 / FREQUENCY_HZ
time_cycle = np.linspace(0.0, period, N_POINTS)
torque = instantaneous_torque(
    time_cycle, PHASE, em.drive_angular_frequency, ROTOR_SPEED_FOR_TORQUE,
    CONDUCTIVITY, FIELD_1, FIELD_2,
)
average_torque = float(cycle_average_torque(
    PHASE, ROTOR_SPEED_FOR_TORQUE, CONDUCTIVITY, FIELD_1, FIELD_2
))
plot_torque_cycle(em.drive_angular_frequency * time_cycle, torque, average_torque);
print('Cycle-average torque:', average_torque, 'N m')

In [ ]:
terminal_speed = float(reduced_terminal_speed(
    PHASE, factor.real, m['disk_1_disk_2_only'].area, m['triple'].area,
    em.drive_angular_frequency,
))
time = np.linspace(0.0, FINAL_TIME, N_POINTS)
rotor = normalized_rotor(terminal_speed, TIME_CONSTANT)
plot_transient(time, rotor.angular_speed(time), terminal_speed);
print('Reduced terminal speed:', terminal_speed, 'rad/s')
if geometry_parameters != GeometryParameters():
    print('WARNING: recompute the torque coefficients for this custom geometry.')